# Artificial Intelligence Assignment 1
## Telco Customer Churn Prediction

This notebook follows the assignment workflow: load and understand the dataset, clean the data, perform exploratory data analysis, prepare features, train two classification models, evaluate and compare them, and predict churn for a sample customer.

**Dataset:** `WA_Fn-UseC_-Telco-Customer-Churn.csv`

Run the cells from top to bottom. When prompted, upload the CSV file in Google Colab.

## 1. Import libraries and upload the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

uploaded = files.upload()
file_name = next(iter(uploaded))
data = pd.read_csv(file_name)

data.head()

## 2. Understand the dataset
Check the number of rows and columns, column names, data types, and basic numerical statistics.

In [ ]:
print("Shape:", data.shape)

print("\nColumns:")
print(data.columns.tolist())

print("\nData Types:")
print(data.dtypes)

print("\nBasic Statistics:")
print(data.describe())

## 3. Check data quality
The `TotalCharges` column may contain blank strings. These are not initially counted as missing values, so check them separately.

In [ ]:
print("Missing Values:")
print(data.isnull().sum())

print("\nDuplicate Rows:")
print(data.duplicated().sum())

print("\nBlank TotalCharges:")
print((data["TotalCharges"].astype(str).str.strip() == "").sum())

## 4. Clean the dataset
Convert `TotalCharges` to numeric values, remove rows with missing values, remove duplicates, and drop the customer ID because it is an identifier rather than a customer behavior feature.

In [ ]:
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

print("Missing Values After Conversion:")
print(data.isnull().sum())

data = data.dropna()
data = data.drop_duplicates()
data = data.drop("customerID", axis=1)

print("\nShape After Cleaning:", data.shape)

## 5. Exploratory data analysis
### Churn distribution
Display the number of customers who stayed and the number who left.

In [ ]:
print(data["Churn"].value_counts())

data["Churn"].value_counts().plot(kind="bar", title="Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()

### Churn by contract type
Compare the churn counts across contract types.

In [ ]:
pd.crosstab(data["Contract"], data["Churn"]).plot(kind="bar")
plt.title("Churn by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()

### Monthly charges by churn
Use boxplots to compare monthly charges for customers who stayed and customers who churned.

In [ ]:
data.boxplot(column="MonthlyCharges", by="Churn")
plt.title("Monthly Charges by Churn")
plt.suptitle("")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.show()

## 6. Prepare features and target
`X` contains the input features. `y` contains the target. Encode the target as 0 for No and 1 for Yes, and convert categorical input columns into numeric indicator columns.

In [ ]:
X = data.drop("Churn", axis=1)

y = data["Churn"].map({
    "No": 0,
    "Yes": 1
})

X = pd.get_dummies(X, dtype=int)

print("X Shape:", X.shape)
print("y Shape:", y.shape)
X.head()

## 7. Split training and testing data
Use 80% of the data for training and 20% for testing. Stratification keeps the churn class proportions similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

## 8. Scale features for Logistic Regression
Fit the scaler using only the training data, then use the same scaler to transform the test data. The Decision Tree will use the unscaled features.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 9. Train Logistic Regression

In [ ]:
logistic_model = LogisticRegression(max_iter=1000)

logistic_model.fit(X_train_scaled, y_train)

logistic_predictions = logistic_model.predict(X_test_scaled)

## 10. Train Decision Tree

In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)

## 11. Evaluate both models
The evaluation reports Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and the classification report.

In [ ]:
def evaluate_model(name, actual, predicted):
    print("\n", name)
    print("Accuracy:", round(accuracy_score(actual, predicted), 4))
    print("Precision:", round(precision_score(actual, predicted), 4))
    print("Recall:", round(recall_score(actual, predicted), 4))
    print("F1-Score:", round(f1_score(actual, predicted), 4))
    print("Confusion Matrix:")
    print(confusion_matrix(actual, predicted))
    print("Classification Report:")
    print(classification_report(actual, predicted))

evaluate_model("Logistic Regression", y_test, logistic_predictions)
evaluate_model("Decision Tree", y_test, tree_predictions)

## 12. Compare and select a model
For this assignment, compare the F1-scores and select the model with the higher F1-score. The scores are close, so also consider the precision and recall results when discussing the models.

In [ ]:
logistic_f1 = f1_score(y_test, logistic_predictions)
tree_f1 = f1_score(y_test, tree_predictions)

if logistic_f1 >= tree_f1:
    final_model = logistic_model
    final_model_name = "Logistic Regression"
else:
    final_model = tree_model
    final_model_name = "Decision Tree"

print("Logistic Regression F1-Score:", round(logistic_f1, 4))
print("Decision Tree F1-Score:", round(tree_f1, 4))
print("Selected Final Model:", final_model_name)

## 13. Create a sample customer
This is an example customer record. You can change the values to experiment with different customer profiles.

In [ ]:
sample_customer = pd.DataFrame([{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 85.50,
    "TotalCharges": 1026.00
}])

sample_customer

## 14. Prepare the sample and predict
Apply the same dummy encoding and align the columns with the training features. Then predict using the selected model.

In [ ]:
sample_customer = pd.get_dummies(sample_customer, dtype=int)
sample_customer = sample_customer.reindex(columns=X.columns, fill_value=0)

if final_model_name == "Logistic Regression":
    sample_customer_scaled = scaler.transform(sample_customer)
    sample_prediction = final_model.predict(sample_customer_scaled)[0]
else:
    sample_prediction = final_model.predict(sample_customer)[0]

if sample_prediction == 1:
    print("Prediction: Churn")
else:
    print("Prediction: No Churn")

## 15. Conclusion

In your own words, summarize:
- How many records remained after cleaning.
- What the exploratory graphs show.
- The test metrics for both models.
- Which model was selected using the comparison rule.
- The prediction for the sample customer.

Use the actual outputs from your run when writing your conclusion. Do not copy expected numbers if your output differs.